In [39]:
import pandas as pd

In [40]:
matches = pd.read_csv("matchesSemiFinals.csv", index_col=0)

In [41]:
matches = matches.drop(columns=["opp formation"])

In [42]:
matches.head()

,team,opponent,date,venue,time,comp,round,day,result,gf,...,referee,match report,notes,sh,sot,dist,fk,pk,pkatt,season
0,America,Tijuana,2025-01-16,Home,20:10,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,Puebla,Atletico,2025-01-17,Home,19:00,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,UANL,Mazatlan,2025-01-17,Home,21:00,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,Necaxa,Guadalajara,2025-01-17,Home,21:05,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,Queretaro,Pumas UNAM,2025-01-19,Home,17:00,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [43]:
matches["team"].value_counts()

team
America          72
Monterrey        64
UANL             63
Cruz Azul        62
Pumas UNAM       61
Atletico         61
Guadalajara      59
Toluca           57
Pachuca          56
Tijuana          56
Atlas            55
Puebla           55
Necaxa           55
Leon             55
Queretaro        54
Santos Laguna    54
FC Juarez        53
Mazatlan         53
Name: count, dtype: int64

In [44]:
matches[matches["team"] == "Cruz Azul"].sort_values("date")

,team,opponent,date,venue,time,comp,round,day,result,gf,...,referee,match report,notes,sh,sot,dist,fk,pk,pkatt,season
648,Cruz Azul,Atlas,2023-07-01,Away,22:10,Liga MX,Apertura 2023 Regular Season,Sat,L,0,...,Víctor Cáceres,Match Report,NaN,10.0,3.0,20.2,1.0,0.0,0.0,2023.0
649,Cruz Azul,Toluca,2023-07-08,Home,18:50,Liga MX,Apertura 2023 Regular Season,Sat,L,0,...,Ismael Lopez,Match Report,NaN,11.0,4.0,18.9,0.0,0.0,0.0,2023.0
650,Cruz Azul,Tijuana,2023-07-14,Away,20:10,Liga MX,Apertura 2023 Regular Season,Fri,L,1,...,Edgar Morales,Match Report,NaN,12.0,1.0,21.1,2.0,0.0,0.0,2023.0
651,Cruz Azul,Santos,2023-08-20,Home,18:00,Liga MX,Apertura 2023 Regular Season,Sun,D,2,...,Marco Antonio Ortíz,Match Report,NaN,17.0,7.0,18.6,0.0,1.0,1.0,2023.0
652,Cruz Azul,Pachuca,2023-08-23,Away,20:06,Liga MX,Apertura 2023 Regular Season,Wed,L,0,...,Jesus Lopez,Match Report,NaN,17.0,7.0,20.4,0.0,0.0,0.0,2023.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
26,Cruz Azul,Tijuana,2024-11-27,Away,19:10,Liga MX,Apertura — Quarter-finals,Wed,L,0.0,...,Fernando Hernández,Match Report,Leg 1 of 2,8.0,0.0,19.7,0.0,0.0,0.0,2024.0
27,Cruz Azul,Tijuana,2024-11-30,Home,19:10,Liga MX,Apertura — Quarter-finals,Sat,W,3.0,...,Víctor Cáceres,Match Report,Leg 2 of 2; Cruz Azul won,28.0,9.0,16.0,0.0,0.0,0.0,2024.0
28,Cruz Azul,América,2024-12-05,Away,20:00,Liga MX,Apertura — Semi-finals,Thu,D,0.0,...,Oscar Mejía,Match Report,Leg 1 of 2,7.0,2.0,19.9,0.0,0.0,0.0,2024.0
29,Cruz Azul,América,2024-12-08,Home,19:00,Liga MX,Apertura — Semi-finals,Sun,L,3.0,...,Adonai Escobedo,Match Report,Leg 2 of 2; América won,16.0,5.0,16.4,1.0,0.0,0.0,2024.0


In [45]:
del matches["comp"]

In [46]:
del matches["notes"]

In [47]:
matches["date"] = pd.to_datetime(matches["date"])

In [48]:
matches["target"] = (matches["result"] == "W").astype("int")

In [49]:
matches["venue_code"] = matches["venue"].astype("category").cat.codes

In [50]:
matches["opp_code"] = matches["opponent"].astype("category").cat.codes

In [51]:
matches["hour"] = matches["time"].str.replace(":.+", "", regex=True).astype("int")

In [52]:
matches["day_code"] = matches["date"].dt.dayofweek

In [53]:
from sklearn.ensemble import RandomForestClassifier

In [54]:
rf = RandomForestClassifier(n_estimators=50, min_samples_split=10, random_state=1)

In [77]:
train = matches[matches["date"] < '2025-01-13']
#train = matches[(matches["date"] >= '2024-11-05') & (matches["date"] <= '2024-11-10')]

In [78]:
#test = matches[(matches["date"] >= '2025-01-05') & (matches["date"] <= '2025-01-13')]
test = matches[(matches["date"] >= '2025-01-13')]

In [79]:
predictors = ["venue_code", "opp_code", "hour", "day_code"]

In [80]:
rf.fit(train[predictors], train["target"])

RandomForestClassifier(min_samples_split=10, n_estimators=50, random_state=1)

In [81]:
preds = rf.predict(test[predictors])

In [93]:
preds

array([0, 0, 1, 0, 0, 0, 0, 1, 1])

In [82]:
probabilidades = rf.predict_proba(test[predictors])

In [83]:
df_resultados = pd.DataFrame(probabilidades, columns=rf.classes_)
df_resultados['Prediccion'] = rf.predict(test[predictors])
df_resultados = df_resultados.reset_index(drop=True)
test = test.reset_index(drop=True)
df_resultados['Equipo'] = test['team']
df_resultados['Oponente'] = test['opponent']
df_resultados['Fecha']= test['date']

In [94]:
df_resultados

,0,1,Prediccion,Equipo,Oponente,Fecha
0,0.511906,0.488094,0,America,Tijuana,2025-01-16
1,0.681434,0.318566,0,Puebla,Atletico,2025-01-17
2,0.390928,0.609072,1,UANL,Mazatlan,2025-01-17
3,0.507267,0.492733,0,Necaxa,Guadalajara,2025-01-17
4,0.503439,0.496561,0,Queretaro,Pumas UNAM,2025-01-19
5,0.632581,0.367419,0,Toluca,Monterrey,2025-01-18
6,0.660285,0.339715,0,FC Juarez,Cruz Azul,2025-01-18
7,0.393280,0.606720,1,Atlas,León,2025-01-18
8,0.274104,0.725896,1,Pachuca,Santos Laguna,2025-01-20


Evaluar exactitud

In [95]:
from sklearn.metrics import accuracy_score

In [96]:
accuracy= accuracy_score(test["target"], preds)

In [97]:
accuracy

0.6666666666666666

In [98]:
combined = pd.DataFrame(dict(actual=test["target"], predicted=preds))
pd.crosstab(index=combined["actual"], columns=combined["predicted"])

predicted,0,1
actual,,
0,6,3


In [99]:
from sklearn.metrics import precision_score

precision_score(test["target"], preds)

0.0

In [100]:
# Calcular la precisión de la clase "Gana el equipo local" (1)
precision_local = precision_score(test["venue_code"], preds, pos_label=1)

print("Precisión para las predicciones de 'Gana el equipo local':", precision_local)

Precisión para las predicciones de 'Gana el equipo local': 1.0


Considerar solo los esenarios en que el modelo coincide con el ganador y el perdedor


Calcula El dinero a pagar

In [101]:
# Define la cantidad inicial de dinero
dinero = 400  

In [102]:
# Descarta los partidos predecidos donde el ganador es el visitante
df_resultados[1] = df_resultados[1]*df_resultados['Prediccion']
# Calcula la suma total de las probabilidades de ganar local
suma_probabilidades_local = df_resultados[1].sum()
# Calcula la apuesta para cada fila
df_resultados['Apuesta'] = (df_resultados[1] / suma_probabilidades_local) * dinero
print(df_resultados)

          0         1  Prediccion     Equipo       Oponente      Fecha  \
0  0.511906  0.000000           0    America        Tijuana 2025-01-16   
1  0.681434  0.000000           0     Puebla       Atletico 2025-01-17   
2  0.390928  0.609072           1       UANL       Mazatlan 2025-01-17   
3  0.507267  0.000000           0     Necaxa    Guadalajara 2025-01-17   
4  0.503439  0.000000           0  Queretaro     Pumas UNAM 2025-01-19   
5  0.632581  0.000000           0     Toluca      Monterrey 2025-01-18   
6  0.660285  0.000000           0  FC Juarez      Cruz Azul 2025-01-18   
7  0.393280  0.606720           1      Atlas           León 2025-01-18   
8  0.274104  0.725896           1    Pachuca  Santos Laguna 2025-01-20   

      Apuesta  
0    0.000000  
1    0.000000  
2  125.472658  
3    0.000000  
4    0.000000  
5    0.000000  
6    0.000000  
7  124.988163  
8  149.539180  


In [74]:
grouped_matches = matches.groupby("team")

In [75]:
group = grouped_matches.get_group("Cruz Azul").sort_values("date")

In [35]:
def rolling_averages(group, cols, new_cols):
    group = group.sort_values("date")
    rolling_stats = group[cols].rolling(3, closed='left').mean()
    group[new_cols] = rolling_stats
    group = group.dropna(subset=new_cols)
    return group

In [36]:
cols = ["gf", "ga", "sh", "sot", "dist", "fk", "pk", "pkatt"]
new_cols = [f"{c}_rolling" for c in cols]

rolling_averages(group, cols, new_cols)

,team,opponent,date,venue,time,round,day,result,gf,ga,...,hour,day_code,gf_rolling,ga_rolling,sh_rolling,sot_rolling,dist_rolling,fk_rolling,pk_rolling,pkatt_rolling
634,Cruz Azul,Santos,2023-08-20,Home,18:00,Apertura 2023 Regular Season,Sun,D,2,2,...,18,6,0.333333,2.000000,11.000000,2.666667,20.066667,1.000000,0.000000,0.000000
635,Cruz Azul,Pachuca,2023-08-23,Away,20:06,Apertura 2023 Regular Season,Wed,L,0,1,...,20,2,1.000000,2.000000,13.333333,4.000000,19.533333,0.666667,0.333333,0.333333
636,Cruz Azul,Monterrey,2023-08-27,Away,21:05,Apertura 2023 Regular Season,Sun,W,2,1,...,21,6,1.000000,1.666667,15.333333,5.000000,20.033333,0.666667,0.333333,0.333333
637,Cruz Azul,América,2023-09-02,Home,22:10,Apertura 2023 Regular Season,Sat,L,2,3,...,22,5,1.333333,1.333333,15.666667,6.666667,18.366667,0.000000,0.333333,0.333333
638,Cruz Azul,Mazatlán,2023-09-15,Away,18:50,Apertura 2023 Regular Season,Fri,D,2,2,...,18,4,1.333333,1.666667,13.666667,5.666667,17.500000,0.666667,0.000000,0.000000
639,Cruz Azul,Querétaro,2023-09-24,Home,17:00,Apertura 2023 Regular Season,Sun,L,1,3,...,17,6,2.000000,2.000000,12.333333,5.333333,15.900000,0.666667,0.000000,0.000000
640,Cruz Azul,Atlético,2023-09-29,Away,22:00,Apertura 2023 Regular Season,Fri,W,2,1,...,22,4,1.666667,2.666667,12.333333,5.666667,15.400000,0.666667,0.000000,0.000000
641,Cruz Azul,Necaxa,2023-10-04,Away,20:00,Apertura 2023 Regular Season,Wed,W,3,1,...,20,2,1.666667,2.000000,11.333333,4.666667,18.166667,0.000000,0.333333,0.333333
642,Cruz Azul,UNAM,2023-10-07,Home,22:10,Apertura 2023 Regular Season,Sat,L,1,4,...,22,5,2.000000,1.666667,12.333333,3.666667,20.066667,0.333333,0.666667,0.666667
643,Cruz Azul,UANL,2023-10-21,Away,22:10,Apertura 2023 Regular Season,Sat,L,1,2,...,22,5,2.000000,2.000000,13.000000,3.000000,21.200000,0.666667,0.666667,0.666667


In [37]:
matches_rolling = matches.groupby("team").apply(lambda x: rolling_averages(x, cols, new_cols))

DataError: Cannot aggregate non-numeric type: object

In [ ]:
matches_rolling

In [ ]:
matches_rolling = matches_rolling.droplevel('team')

In [ ]:
matches_rolling.index = range(matches_rolling.shape[0])

In [ ]:
def make_predictions(data, predictors):
    train = data[data["date"] < '2022-01-01']
    test = data[data["date"] > '2022-01-01']
    rf.fit(train[predictors], train["target"])
    preds = rf.predict(test[predictors])
    combined = pd.DataFrame(dict(actual=test["target"], predicted=preds), index=test.index)
    error = precision_score(test["target"], preds)
    return combined, error

In [ ]:
combined, error = make_predictions(matches_rolling, predictors + new_cols)

In [ ]:
error

In [ ]:
combined = combined.merge(matches_rolling[["date", "team", "opponent", "result"]], left_index=True, right_index=True)

In [ ]:
combined.head(10)

In [ ]:
class MissingDict(dict):
    __missing__ = lambda self, key: key

map_values = {"Brighton and Hove Albion": "Brighton", "Manchester United": "Manchester Utd", "Newcastle United": "Newcastle Utd", "Tottenham Hotspur": "Tottenham", "West Ham United": "West Ham", "Wolverhampton Wanderers": "Wolves"} 
mapping = MissingDict(**map_values)

In [ ]:
combined["new_team"] = combined["team"].map(mapping)

In [ ]:
merged = combined.merge(combined, left_on=["date", "new_team"], right_on=["date", "opponent"])

In [ ]:
merged

In [ ]:
merged[(merged["predicted_x"] == 1) & (merged["predicted_y"] ==0)]["actual_x"].value_counts()